# Stormcatalog-analyzer — Function & Output Documentation

*A reference guide to the storm **tracking**, **catalog**, and **plotting** functions.*

This notebook documents **what each function does, its parameters, and the outputs** the tool
produces. It is a reference document: the code cells are **illustrative snippets** (copy-paste to
run) — the actual pipeline lives in the `stormcatalog_analyzer` package.

**Contents**
1. [Storm tracking overview](#1)
2. [Installation](#2)
3. [Repository structure](#3)
4. [Two ways to use Stormcatalog-analyzer (notebook & CLI)](#4)
5. [Storm tracking steps: functions and their parameters](#5)
6. [Repository outputs: catalog table + diagnostic plots](#6)

<a id="1"></a>
## 1. Storm tracking overview

Stormcatalog-analyzer detects, tracks, and characterizes the **motion of storms** from gridded
rainfall *storm catalogs* (a folder of `.nc` events produced by the **RainyDay** package). For each
storm it quantifies **direction, speed, spatial extent (a fitted ellipse), and intensity** through
**Lagrangian tracking of storm centroids**.

The function needs hourly rainfall necCDF file + a JSON config → the tool produces a modular set of
diagnostic plots and a per-storm properties table.

### The pipeline in three stages
1. **Track** — `run_catalog_tracking(cfg)`. Per event: identify storm areas → track them across
   time → select the single longest continuous storm → fit an ellipse and compute the centroid at
   every time step.
2. **Summarize** — `summarize_motion(results, cfg)`. Per storm: find the most-intense time window,
   then compute mean direction, speed, intensity, trajectory, and ellipse properties over it.
3. **Diagnose** — `generate_catalog_diagnostics(summary, cfg)`. Write the per-storm CSV and the
   catalog-level figures.

`pipeline.run(cfg)` chains all three and returns `{"results", "summary", "figures", "table"}`.

### Key concepts
- **Identification** — morphological image processing (threshold → dilation/erosion → connected
  components) turns each time step's rainfall field into labeled storm areas.
- **Tracking** — a STEP-style *overlap-ratio* algorithm links storm areas across consecutive time
  steps into continuous objects.
- **Ellipse fitting** — an ellipse is fit to each step's rainfall, giving major/minor axes and orientation (a proxy for storm size and shape).
- **Centroid motion** — the precipitation-weighted centroid is tracked step to step; its path is the
  storm trajectory, from which direction and speed are derived.
- **Most-intense window** — motion statistics are computed over the `intensity_window` (e.g. `12H`)
  with the highest accumulated in-ellipse rainfall, **not** the whole event to keep track of consistent and continuos storms

### Direction conventions (important)
| Name | Convention | Zero points | Positive rotation |
|------|-----------|-------------|-------------------|
| `direction_deg` | mathematical | East (0°) | counter-clockwise |
| `bearing_deg`   | compass      | North (0°) | clockwise |

`angular_variance = 1 − R` (mean resultant length): **0 = perfectly straight track**, values toward
**1 = erratic / rotating** motion.

<a id="2"></a>
## 2. Installation

**Prerequisites**
- Python 3.7+
- The scientific stack: `numpy, pandas, scipy, xarray, netCDF4, scikit-image, opencv-python, pyproj,
  geographiclib, mpu, geopandas, shapely, cartopy, matplotlib, seaborn, windrose, imageio, Pillow`.

In [ ]:
# Clone and install (shell)
# git clone https://github.com/Perez-HydroSystems/storm-motion-analyzer.git
# cd storm-motion-analyzer
#
# pip install -r requirements.txt
# pip install -e .          # optional: enables the `stormcatalog-analyzer` CLI command

> **Tip (development):** this project is verified against a conda environment that carries the full
> geospatial stack (xarray, geopandas, cartopy, opencv, windrose, …); a bare Python often lacks
> these. For **headless** runs (scripts, servers, CI) set the non-interactive Matplotlib backend so
> figures render without a display:

In [ ]:
import os
os.environ["MPLBACKEND"] = "Agg"   # headless rendering
# equivalently on the shell:  MPLBACKEND=Agg python run_diagnostics.py --config configs/testing_data.json

<a id="3"></a>
## 3. Repository structure

```
stormcatalog_analyzer/          # the installable package (import as `stormcatalog_analyzer`)
  config.py     # Config dataclasses + load_config(json) + validation  (SINGLE source of truth)
  io.py         # discover_events, open_event (xarray), load_vector (geopandas), output dirs
  pipeline.py   # run_catalog_tracking -> summarize_motion -> generate_catalog_diagnostics; run()
  cli.py        # argparse; subcommands `diagnostics`, `event`
  tracking/     # identification.py, tracking.py, catalog.py, dataset.py   (storm-tracking core)
  motion/       # direction.py (per-storm dir/speed), grid.py (storm-motion grid)
  stats/        # circular.py (circular mean / percentile / trend / permutation)
  plotting/     # helpers.py, diagnostics.py (catalog figures + pairplot),
                #   per_event.py (track / time-steps), motion_grid.py (grid plots),
                #   animation.py (animate_storm_tracking -> per-storm tracking GIF)
  geometry.py   # control-area shapefile builders
run_diagnostics.py            # thin root CLI wrapper (defaults to `diagnostics`)
configs/testing_data.json     # default config -> relative paths into testing_data/
storm_motion_catalog.ipynb    # MAIN notebook (load config -> pipeline.run -> show figures)
notebooks/                    # storm_motion_event.ipynb + legacy/ (unmaintained)
pyproject.toml  requirements.txt  README.md  CLAUDE.md
```

Paths in the config resolve **relative to `project_root`** — there are no absolute paths in committed
code or configs. The large `testing_data/` catalog is gitignored (outputs land there too).

<a id="4"></a>
## 4. Two ways to use Stormcatalog-analyzer

Both entry points share the **same JSON config** and the **same pipeline** — only how you launch it
differs.

### A) Notebook / Python API

In [ ]:
from stormcatalog_analyzer.config import load_config
from stormcatalog_analyzer import pipeline

cfg = load_config("configs/testing_data.json")
cfg.tracking.rainfall_threshold_mmhr = 0.5     # optional inline overrides
# cfg.io.max_events = 20                        # subsample for a quick look

out = pipeline.run(cfg)                         # track -> summarize -> diagnostics
out["summary"].storm_properties.head()          # per-storm properties table
print(out["table"])                             # path to storm_properties_<domain>.csv
print(out["figures"])                           # saved figure paths

Open **`storm_motion_catalog.ipynb`** for the full notebook: catalog figures, all per-event figures,
the direction-probability summary, and the storm-tracking animation.

### B) Command line (headless)

In [ ]:
# Catalog-level diagnostic plots (main deliverable)
# MPLBACKEND=Agg python run_diagnostics.py --config configs/testing_data.json
#
# Per-event tracking figures for one storm (by filename, substring, or id)
# MPLBACKEND=Agg python run_diagnostics.py event --config configs/testing_data.json --storm 112
#
# Subsample the catalog to the first N events
# python run_diagnostics.py --config configs/testing_data.json --max-events 20
#
# (If installed with `pip install -e .`:  stormcatalog-analyzer diagnostics|event ... )

<a id="5"></a>
## 5. Storm tracking steps: functions and their parameters

### 5.0 Configuration — the single source of truth
All parameters and I/O paths live in **one JSON config**, parsed into dataclasses and validated once
by `config.load_config`. Sections and defaults:

**`io`**

| Key | Default | Meaning |
|-----|---------|---------|
| `catalog_dir` | — | Folder of `.nc` storm events (the catalog) |
| `transposition_domain_path` | — | Domain polygon shapefile |
| `control_area_path` | `null` | Optional control-area outline (e.g. a basin) |
| `grid_path` | `null` | Optional CONUS 5×5 reference grid (paper inset) |
| `output_dir` | `outputs` | Where results are written |
| `domain_name` | `Domain` | Label used in filenames / titles |
| `max_events` | `null` | Process only the first N events (`null` = all) |

**`tracking`**

| Key | Default | Meaning |
|-----|---------|---------|
| `rainfall_var_name` | `"rain"` | Name of the rain variable in the NetCDF |
| `morph_radius_cells` | `2` | Morphological merge radius (grid cells); larger merges nearby cells into one storm |
| `rainfall_threshold_mmhr` | `0.5` | Cells below this (mm/h) are dropped before identification |
| `ellipse_fit_method` | `"moments"` | Ellipse fit: `"moments"` or `"contour"` |
| `overlap_ratio_threshold` | `0.2` | Min overlap ratio to link storms across time *(config-only, see note)* |
| `dry_spell_steps` | `0` | Allowed gap when matching storms *(config-only, see note)* |

**`selection`**

| Key | Default | Meaning |
|-----|---------|---------|
| `min_duration_steps` | `6` | Drop storms whose longest tracked duration < this (time steps) |
| `min_area_fraction` | `0.05` | Storm must cover ≥ this fraction of the domain *(config-only, see note)* |

**`motion`**

| Key | Default | Meaning |
|-----|---------|---------|
| `intensity_window` | `"12H"` | Window for the most-intense period |
| `n_trajectories_plotted` | `100` | Number of (longest) trajectories drawn on the map |

**`figure`** — `font_size`, `dpi`, `smooth_factor` (trajectory spline), and arrow styling
(`arrow_scale`, `arrow_width`, `head_width`, `head_length`) for the direction-vector plot.

**`direction_grid`** — `n_sectors` (e.g. 4/8), `cell_size_km`, `count_threshold`, `start_angle_deg`
for the storm-motion direction-probability diagnostic.

### 5.1 Stage 1 — tracking (`run_catalog_tracking`)
For each event it calls three functions in order (from `tracking/dataset.py`):

**`storm_tracking_event(rain_dataset, storm_dict, morph_radius, high_threshold, var_name)`**
Reads the rainfall array; runs identification → tracking → longest-event selection; stores the
labeled arrays, coordinate grids (geographic **and** projected EPSG:2163), and `longest_duration`.
Config wiring: `morph_radius ← tracking.morph_radius_cells`, `high_threshold ← rainfall_threshold_mmhr`,
`var_name ← rainfall_var_name`. Internally:
- `storm_area_identification(prcp, morph_radius, high_threshold)` — threshold → dilation/erosion →
  8-connectivity connected components → labeled storm areas per step.
- `track(identification_array, ratio_threshold=0.2, dry_spell_time=0)` — STEP overlap-ratio tracking;
  links areas whose combined (t→t-1 + t-1→t) overlap ratio exceeds `ratio_threshold`.
- `single_event_selection(track_array, area_portion_threshold=0.05)` — pick the storm label with the
  longest duration (steps where its area > 5% of the domain).

**`continuos_storm(storm_dict)`**
Extracts the continuous block of time steps where the selected storm's area exceeds 5% of the
domain; stores `selected_storm` (rain), `selected_storm_area`, `selected_storm_time_steps`.

**`storm_tracking_features(storm_dict, ellipse_fit)`**
Per time step computes the precipitation-weighted **centroid**, the area-weighted **avg** & **max**
rainfall, a fitted **ellipse** (major/minor axes, angle, centroid), and the step **bearing** +
**distance**. Builds the per-step `storm_record` DataFrame (Section 6.1b).
Config wiring: `ellipse_fit ← tracking.ellipse_fit_method` (moments unless *exactly* `"contour"`).

> After tracking, an event is **kept** only if `longest_duration ≥ selection.min_duration_steps`.

### 5.2 Stage 2 — motion summary (`summarize_motion` → `compute_storm_trajectory`)
Per retained storm, over its **most-intense window**:

| Function (`motion/direction.py`) | Returns | Notes |
|---|---|---|
| `mean_precipitation_within_ellipse(event_dict)` | in-ellipse mean rain per step | mask from the fitted ellipse |
| `get_max_accumulated_value(series, window)` | max sum + start/end index | `window ← motion.intensity_window` |
| `mean_velocity(x, y)` | mean speed (m/s) | mean centroid step distance / 3600 |
| `mean_direction(x, y)` | (mean angle, variance) | unweighted circular mean of segment angles |
| `mean_direction_weighted(x, y)` | (mean angle, variance) | segment-length-weighted circular mean |
| `compute_line_length(x, y)` | path length (m) | sum of segment lengths |

`x, y` are the **projected** centroids over the window. These metrics are written back onto the
event_dict (the per-event plots read them there) and returned as the CSV row. Catalog-level
aggregates (mean angle/speed/intensity) and geographic trajectories (EPSG:2163→4326) are assembled
here too.

### 5.3 Stage 3 — diagnostics (`generate_catalog_diagnostics`)
Writes `storm_properties_<domain>.csv` and the three catalog figures (Section 6).

> **Note — config-only parameters (current state):** `overlap_ratio_threshold`, `dry_spell_steps`,
> and `min_area_fraction` are validated and stored in the config, but the low-level tracking
> functions currently use their **internal defaults** (`ratio_threshold=0.2`, `dry_spell_time=0`,
> `area_portion_threshold=0.05`). Editing them in the JSON does not yet change tracking.

<a id="6"></a>
## 6. Repository outputs

Everything is written under **`output_dir/<domain_name>/`**.

### 6.1 Catalog table — `storm_properties_<domain>.csv`
**One row per storm**, computed over its most-intense window:

| Column | Unit | Description |
|--------|------|-------------|
| `storm_event` | — | Catalog filename of the event |
| `storm_id` | — | Storm id parsed from the filename |
| `start_time` | datetime | Start of the most-intense window |
| `end_time` | datetime | End of the most-intense window |
| `duration_hours` | steps | Length of the most-intense window |
| `mean_speed_ms` | m/s | Mean centroid translation speed (`mean_velocity`) |
| `mean_direction_deg` | ° (math) | Weighted mean direction (`mean_direction_weighted`; 0=E, CCW+) |
| `mean_direction_vector_deg` | ° (math) | Unweighted mean direction (`mean_direction`) |
| `angular_variance` | 0–1 | `1 − R`; 0 = straight track, →1 = erratic |
| `mean_intensity_mmh` | mm/h | Mean in-ellipse rainfall over the window |
| `peak_intensity_mmh` | mm/h | Max per-step maximum rainfall over the window |
| `total_rainfall_mm` | mm | Sum of in-ellipse mean rainfall over the window |
| `storm_area_km2` | km² | Mean storm area over the window |
| `trajectory_length_km` | km | Path length of the centroid track |
| `mean_major_axis_km` | km | Mean fitted-ellipse major axis |
| `mean_minor_axis_km` | km | Mean fitted-ellipse minor axis |
| `mean_ellipse_angle_deg` | ° | Mean ellipse orientation |

### 6.1b Per-step record — `storm_record` (in memory, per event)
Behind each storm is a `storm_record` DataFrame with **one row per time step** — the raw tracked
series the summary is built from:

| Column | Unit | Description |
|--------|------|-------------|
| `time`, `year`, `month` | — | Time-step timestamp and its year/month |
| `duration(hour)` | h | Number of steps in the selected storm |
| `avg_prcp(mm/h)` | mm/h | Area-weighted mean rainfall at the step |
| `max_prcp(mm/h)` | mm/h | Maximum rainfall at the step |
| `cent_lon(degree)`, `cent_lat(degree)` | ° | Precip-weighted centroid (geographic) |
| `cen_prj_lon(m)`, `cen_prj_lat(m)` | m | Centroid in the projected CRS (EPSG:2163) |
| `bearing(degree)` | ° (compass) | Bearing between consecutive centroids |
| `distance(km)` | km | Haversine distance between consecutive centroids |
| `major_axis_length(m)`, `minor_axis_length(m)` | m | Fitted-ellipse axes |
| `ellipse_angle (degree)` | ° | Ellipse orientation (major axis vs x, clockwise) |
| `ellipse_cent_prj_lon(m)` / `_lat(m)` | m | Ellipse centroid (projected) |
| `ellipse_cent_lon` / `_lat` | ° | Ellipse centroid (geographic) |
| `matplotlib_width(m)`, `matplotlib_height(m)` | m | Ellipse horizontal/vertical extents (for plotting) |

### 6.2 Diagnostic plots

Catalog figures land in `output_dir/<domain>/`; per-event figures in the subfolders `StormTrack/`,
`StormTimeSteps/`, and `StormAnimation/`.

**a) `storm_trajectories_diagnostic_plot_<domain>.png`** — `diagnostic_plot_storm_trajectories`
- **Main map:** the N longest **storm trajectories** (spline-smoothed by `figure.smooth_factor`),
  colored by **mean intensity (mm/h)**; transposition domain (green dashed), optional control area
  (red), US state borders.
- **Wind rose 1:** **storm speed (m/s)** by direction. **Wind rose 2:** **mean intensity (mm/h)** by
  direction (both meteorological convention).
- **Text box:** mean direction, mean speed, mean intensity. **Inset:** domain location on CONUS.

**b) `storm_direction_vector_plot_<domain>.png`** — `diagnostic_plot_storm_direction_vectors`
- **Main map:** **one mean-direction vector per storm** (length = first→last centroid distance),
  colored by **mean intensity** or **angular variance** (`colorbar_var`). Arrow styling comes from
  `figure.*` (`arrow_scale`, `arrow_width`, `head_width`, `head_length`).
- Same two wind roses + a text box (adds **mean angular variance**) + the CONUS inset.

**c) `storm_statistics_pairplot_<domain>.png`** — `plot_storm_properties_pairplot`
- Seaborn pairplot (histograms on the diagonal) across `mean_direction_deg`, `mean_speed_ms`,
  `storm_area_km2`, `mean_intensity_mmh`, `total_rainfall_mm`, `angular_variance`.

**d) `direction_probability_summary_<name>.png`** — `plot_direction_probability_summary`
- Panels: **(a)** total directional counts, **(b)** transposition-domain location, **(c)** per-sector
  direction-probability fields, **(d)** the direction-sector wheel. Sector count comes from
  `direction_grid.n_sectors` (4 → quadrants, 8 → 45°, …); the layout adapts automatically.

**e) Per-event figures**
- `StormTrack/storm_track_<id>.png` — `plot_storm_track`: accumulated rainfall + centroid trajectory
  + fitted ellipses + mean-direction vector + a stats box.
- `StormTimeSteps/storm_step_<id>.png` — `plot_storm_time_steps`: one panel per time step with the
  rainfall field, the centroid, and the fitted ellipse.

**f) Storm-tracking animation** — `StormAnimation/storm_tracking_<id>.gif`
(`animate_storm_tracking` / `pipeline.generate_event_animation`)
- Three phases: **storm moving** → **tracking** (current-step ellipse + growing centroid trail) →
  **trajectory + mean-direction vector + statistics box** (direction, speed, intensity, angular
  variance).

In [ ]:
# Regenerate every output for a small subset (illustrative)
from stormcatalog_analyzer.config import load_config
from stormcatalog_analyzer import pipeline

cfg = load_config("configs/testing_data.json")
cfg.io.max_events = 20                                   # quick look
out = pipeline.run(cfg)                                   # -> CSV + catalog figures (a, b, c)

# per-event track + time-step panels for one storm (e, saved under StormTrack/ & StormTimeSteps/)
pipeline.generate_event_diagnostics(cfg, storm="112")

# storm-tracking animation GIF for one storm (f, saved under StormAnimation/)
pipeline.generate_event_animation(cfg, storm="112", fps=4)